# Session 36 Window Functions in SQL

## What are Window Functions?

Window Functions in SQL are analytical operations that perform calculations across a set of rows that are related to the current row. This set of related rows is called a **window**.

Unlike `GROUP BY`, window functions **do not collapse multiple rows into a single row**. Instead, they calculate a value for each row while retaining the original rows in the result set.

The window over which the calculation is performed is defined using the `OVER()` clause. The `OVER()` clause can specify:

* **Partitioning**: Defines the groups of rows over which the calculation is performed using `PARTITION BY`.
* **Ordering**: Defines the order of rows within each partition using `ORDER BY`.
* **Frame**: Defines a more specific subset of rows within the partition, using clauses such as `ROWS` or `RANGE`.

The basic syntax is:

```sql
window_function()
OVER (
    PARTITION BY column
    ORDER BY column
);
```

### Window Functions vs. GROUP BY

The fundamental difference is:

> **`GROUP BY` reduces rows; Window Functions retain rows.**

Consider a table named `marks`:

| student_id | student_name | branch | marks |
| ---------: | ------------ | ------ | ----: |
|          1 | A            | CSE    |    80 |
|          2 | B            | CSE    |    70 |
|          3 | C            | CSE    |    90 |
|          4 | D            | ECE    |    60 |
|          5 | E            | ECE    |    80 |

Suppose we want to calculate the average marks of each branch.

Using `GROUP BY`:

```sql
SELECT branch,
       AVG(marks) AS branch_avg
FROM marks
GROUP BY branch;
```

Result:

| branch | branch_avg |
| ------ | ---------: |
| CSE    |         80 |
| ECE    |         70 |

Here, the five original rows have been reduced to two rows. Individual student information is no longer present in the result.

The same analysis can be performed using a window function:

```sql
SELECT *,
       AVG(marks) OVER(PARTITION BY branch) AS branch_avg
FROM marks;
```

Result:

| student_id | student_name | branch | marks | branch_avg |
| ---------: | ------------ | ------ | ----: | ---------: |
|          1 | A            | CSE    |    80 |         80 |
|          2 | B            | CSE    |    70 |         80 |
|          3 | C            | CSE    |    90 |         80 |
|          4 | D            | ECE    |    60 |         70 |
|          5 | E            | ECE    |    80 |         70 |

The branch average is calculated independently for each branch, but **every student's original row is preserved**.

This makes window functions particularly useful for analytical queries where we need to compare an individual row with its group, calculate rankings, running totals, percentages, deviations from averages, and other row-level metrics.

### Important Note on Redundancy

Window functions can intentionally produce repeated values in a result set. For example, the CSE average of `80` appears for every CSE student.

This repetition should not be interpreted as a violation of database normalization or Codd's rules. **Normalization concerns how data is stored and structured in relations, whereas a query result is a derived representation of that data.**

Therefore, window functions should be viewed primarily as an **analytical querying technique**, rather than as a mechanism for storing redundant data.

## Aggregate Operations

Aggregate functions such as `AVG()`, `SUM()`, `MIN()`, `MAX()`, and `COUNT()` normally operate on a set of rows and return a summarized result.

For example:

```sql
SELECT AVG(marks) AS average_marks
FROM marks;
```

This returns a single value representing the average marks of all students.

However, an aggregate function can also be used as a window function by combining it with the `OVER()` clause:

```sql
SELECT *,
       AVG(marks) OVER() AS overall_avg
FROM marks;
```

Result:

| student_id | student_name | branch | marks | overall_avg |
| ---------: | ------------ | ------ | ----: | ----------: |
|          1 | A            | CSE    |    80 |          76 |
|          2 | B            | CSE    |    70 |          76 |
|          3 | C            | CSE    |    90 |          76 |
|          4 | D            | ECE    |    60 |          76 |
|          5 | E            | ECE    |    80 |          76 |

The average is calculated over the entire result set because no `PARTITION BY` clause has been specified.

The key difference is:

```sql
AVG(marks)
```

produces an aggregate result, whereas:

```sql
AVG(marks) OVER()
```

produces an analytical value **for every row**.

### Partitioning with `PARTITION BY`

When analysis needs to be performed separately for different groups, `PARTITION BY` can be used inside `OVER()`.

```sql
SELECT *,
       AVG(marks) OVER(PARTITION BY branch) AS branch_avg
FROM marks;
```

Here, the rows are logically divided into partitions based on `branch`.

For example:

```text
CSE Partition
80
70
90

ECE Partition
60
80
```

The average is then calculated independently within each partition.

Therefore:

```sql
AVG(marks) OVER(PARTITION BY branch)
```

means:

> Calculate the average marks independently for each branch and return that average alongside every row belonging to that branch.

### Important Distinction: Aggregate Function vs. Window Function

`AVG()`, `SUM()`, `MIN()`, `MAX()`, and `COUNT()` are **aggregate functions** by definition.

They become capable of performing windowed calculations when they are used with the `OVER()` clause.

For example:

```sql
AVG(marks)
```

is an aggregate operation, while:

```sql
AVG(marks) OVER(PARTITION BY branch)
```

is an aggregate function being used in a **windowed context**.

Therefore, it is more accurate to say:

> **`OVER()` transforms the way an aggregate function operates by allowing it to perform calculations across a window without collapsing the result set.**

## Examples Using `OVER()`

### Minimum and Maximum Marks by Branch

```sql
SELECT *,
       MIN(marks) OVER(PARTITION BY branch) AS min_by_branch,
       MAX(marks) OVER(PARTITION BY branch) AS max_by_branch
FROM marks
ORDER BY student_id;
```

This query calculates the minimum and maximum marks within each branch while retaining every student's individual record.

For example:

| student_id | branch | marks | min_by_branch | max_by_branch |
| ---------: | ------ | ----: | ------------: | ------------: |
|          1 | CSE    |    80 |            70 |            90 |
|          2 | CSE    |    70 |            70 |            90 |
|          3 | CSE    |    90 |            70 |            90 |
|          4 | ECE    |    60 |            60 |            80 |
|          5 | ECE    |    80 |            60 |            80 |

This allows us to compare an individual student's marks against the performance range of their respective branch.

### Finding Students Scoring Above Their Branch Average

> Find all students who have marks higher than the average marks of their respective branch.

A window function can calculate the branch average first, after which the result can be filtered:

```sql
SELECT *
FROM (
    SELECT *,
           AVG(marks) OVER(PARTITION BY branch) AS branch_avg
    FROM marks
) AS student_marks
WHERE marks > branch_avg;
```

The inner query calculates the average for each branch while retaining individual student records.

The outer query then filters students whose marks are greater than their respective branch average.

This illustrates one of the major strengths of window functions:

> **They allow aggregate-level information to be brought down to the individual-row level, making row-to-group comparisons straightforward.**

## Actual Window Functions

Unlike aggregate functions such as `AVG()`, `SUM()`, and `MAX()`, which can be used as window functions through `OVER()`, SQL also provides functions that are specifically designed to perform window-based analysis.

The most commonly used ranking window functions are:

* `RANK()`
* `DENSE_RANK()`
* `ROW_NUMBER()`

These functions are particularly useful for ranking records, identifying top-N records, assigning sequential numbers, and performing group-wise analysis.

### RANK(), DENSE_RANK(), and ROW_NUMBER()

#### RANK()

`RANK()` assigns a rank to each row based on the specified ordering.

```sql
SELECT *,
       RANK() OVER(ORDER BY marks DESC) AS rank
FROM marks;
```

For example:

| student_id | name    | branch | marks | rank |
| :--------- | :------ | :----- | ----: | ---: |
| 7          | Deepak  | CSE    |    98 |    1 |
| 12         | Rohit   | ECE    |    95 |    2 |
| 9          | Vinay   | ECE    |    95 |    2 |
| 8          | Arpan   | CSE    |    95 |    2 |
| 2          | Rishabh | EEE    |    91 |    5 |
| 10         | Ankit   | ECE    |    88 |    6 |

The important characteristic of `RANK()` is its handling of **ties**.

When multiple rows have the same value, they receive the same rank. However, the next rank accounts for the number of tied rows.

In the above example:

```text
98 → Rank 1
95 → Rank 2
95 → Rank 2
95 → Rank 2
91 → Rank 5
88 → Rank 6
```

There are three students occupying rank `2`. Therefore, the next student receives rank `5`, not rank `3`.

This is commonly referred to as **competition ranking**.

> **RANK() leaves gaps in the ranking sequence whenever ties occur.**

### Ranking Within Groups

Ranking does not necessarily have to be performed across the entire result set. We can use `PARTITION BY` to independently rank rows within each group.

```sql
SELECT *,
       RANK() OVER(
           PARTITION BY branch
           ORDER BY marks DESC
       ) AS branch_rank
FROM marks;
```

Here, a separate ranking sequence is created for every `branch`.

For example:

```text
CSE
98 → 1
95 → 2
78 → 3

ECE
95 → 1
95 → 1
88 → 3

EEE
91 → 1
82 → 2
69 → 3
```

The ranking therefore **restarts whenever a new partition begins**.

This combination of `PARTITION BY` and ranking functions is particularly important in analytical SQL because it allows us to answer questions such as:

* Who are the top 3 students in each branch?
* Who are the highest-paying customers in each month?
* What are the top-performing products in each category?
* What is the ranking of each employee within their department?

---

#### DENSE_RANK()

`DENSE_RANK()` is similar to `RANK()` because rows with the same ordering value receive the same rank.

The difference is that `DENSE_RANK()` **does not leave gaps after ties**.

Consider the following query:

```sql
SELECT *,
       RANK() OVER(
           PARTITION BY branch
           ORDER BY marks DESC
       ) AS rank,
       DENSE_RANK() OVER(
           PARTITION BY branch
           ORDER BY marks DESC
       ) AS dense_rank
FROM marks;
```

Output:

| student_id | name    | branch | marks | rank | dense_rank |
| :--------- | :------ | :----- | ----: | ---: | ---------: |
| 7          | Deepak  | CSE    |    98 |    1 |          1 |
| 8          | Arpan   | CSE    |    95 |    2 |          2 |
| 5          | Shubham | CSE    |    78 |    3 |          3 |
| 6          | Ved     | CSE    |    43 |    4 |          4 |
| 9          | Vinay   | ECE    |    95 |    1 |          1 |
| 12         | Rohit   | ECE    |    95 |    1 |          1 |
| 10         | Ankit   | ECE    |    88 |    3 |          2 |

> **Observe the last three rows.**

For the ECE branch:

```text
Marks        RANK()        DENSE_RANK()
95           1             1
95           1             1
88           3             2
```

Both `RANK()` and `DENSE_RANK()` recognise that the two students with `95` marks are tied and therefore assign them the same rank.

However, their treatment of the next student differs:

* `RANK()` assigns `3` because two rows occupied rank `1`.
* `DENSE_RANK()` assigns `2` because it simply moves to the next distinct ranking position.

Therefore:

> **RANK() → Gaps are possible after ties.**
> **DENSE_RANK() → No gaps occur after ties.**

A useful way to remember the distinction is:

| Function       | Same values get same rank? | Gaps after ties? |
| :------------- | :------------------------: | :--------------: |
| `RANK()`       |             Yes            |        Yes       |
| `DENSE_RANK()` |             Yes            |        No        |
| `ROW_NUMBER()` |             No             |        No        |

---

#### ROW_NUMBER()

`ROW_NUMBER()` assigns a unique sequential number to every row within a window.

Unlike `RANK()` and `DENSE_RANK()`, tied values do **not** receive the same number.

The numbering can be restarted for every partition using `PARTITION BY`.

```sql
SELECT *,
       ROW_NUMBER() OVER(
           PARTITION BY branch
       ) AS row_number
FROM marks;
```

Output:

| student_id | name    | branch | marks | row_number |
| :--------- | :------ | :----- | ----: | ---------: |
| 8          | Arpan   | CSE    |    95 |          1 |
| 5          | Shubham | CSE    |    78 |          2 |
| 6          | Ved     | CSE    |    43 |          3 |
| 7          | Deepak  | CSE    |    98 |          4 |
| 9          | Vinay   | ECE    |    95 |          1 |
| 10         | Ankit   | ECE    |    88 |          2 |
| 11         | Anand   | ECE    |    81 |          3 |
| 12         | Rohit   | ECE    |    95 |          4 |
| 1          | Nitish  | EEE    |    82 |          1 |
| 2          | Rishabh | EEE    |    91 |          2 |
| 3          | Anukant | EEE    |    69 |          3 |
| 4          | Rupesh  | EEE    |    55 |          4 |

The numbering starts again from `1` whenever a new `branch` partition is encountered.

However, there is an important point here:

> **Without an `ORDER BY` inside `OVER()`, the order in which `ROW_NUMBER()` assigns numbers is not guaranteed.**

Therefore, if we want a deterministic and meaningful sequence, we should specify an ordering criterion.

For example:

```sql
SELECT *,
       ROW_NUMBER() OVER(
           PARTITION BY branch
           ORDER BY student_id
       ) AS row_number
FROM marks;
```

Now the students within each branch are numbered according to `student_id`.

### Creating Custom Group-wise Identifiers

`ROW_NUMBER()` can also be combined with string functions to generate custom identifiers.

For example:

```sql
SELECT *,
       CONCAT(
           branch,
           '-',
           ROW_NUMBER() OVER(PARTITION BY branch)
       ) AS custom_id
FROM marks;
```

This can produce:

| student_id | name    | branch | marks | custom_id |
| :--------- | :------ | :----- | ----: | :-------- |
| 8          | Arpan   | CSE    |    95 | CSE-1     |
| 5          | Shubham | CSE    |    78 | CSE-2     |
| 6          | Ved     | CSE    |    43 | CSE-3     |
| 7          | Deepak  | CSE    |    98 | CSE-4     |
| 9          | Vinay   | ECE    |    95 | ECE-1     |
| 10         | Ankit   | ECE    |    88 | ECE-2     |
| 11         | Anand   | ECE    |    81 | ECE-3     |
| 12         | Rohit   | ECE    |    95 | ECE-4     |
| 1          | Nitish  | EEE    |    82 | EEE-1     |
| 2          | Rishabh | EEE    |    91 | EEE-2     |
| 3          | Anukant | EEE    |    69 | EEE-3     |
| 4          | Rupesh  | EEE    |    55 | EEE-4     |

However, the numbering order should not be left implicit. For a predictable identifier, specify an `ORDER BY` clause:

```sql
SELECT *,
       CONCAT(
           branch,
           '-',
           ROW_NUMBER() OVER(
               PARTITION BY branch
               ORDER BY student_id
           )
       ) AS custom_id
FROM marks;
```

Now, the numbering is deterministic and follows `student_id` within each branch.

This technique can be useful for generating **group-wise sequence numbers** or temporary analytical identifiers.

## Practical Examples

### Finding the Top 2 Paying Customers for Each Month

Suppose an `orders` table contains:

* `date` — date of the order
* `user_id` — customer identifier
* `amount` — order amount

The objective is:

> Find the top 2 customers based on their total spending for each month.

A possible solution is:

```sql
SELECT *
FROM (
    SELECT
        TO_CHAR(date, 'Month') AS month_name,
        user_id,
        SUM(amount) AS total_amount,
        RANK() OVER (
            PARTITION BY TO_CHAR(date, 'Month')
            ORDER BY SUM(amount) DESC
        ) AS month_rank
    FROM orders
    GROUP BY TO_CHAR(date, 'Month'), user_id
    ORDER BY TO_CHAR(date, 'Month')
) AS t
WHERE t.month_rank < 3
ORDER BY month_name DESC, month_rank ASC;
```

Output:

| month_name | user_id | total_amount | month_rank |
| :--------- | ------: | -----------: | ---------: |
| May        |       1 |          965 |          1 |
| May        |       3 |          860 |          2 |
| June       |       2 |         1480 |          1 |
| June       |       4 |          800 |          2 |
| July       |       5 |         3035 |          1 |
| July       |       2 |         1190 |          2 |

### Understanding the Query

#### 1. Aggregate spending for each customer

```sql
SUM(amount) AS total_amount
```

We need to determine how much each customer spent in total.

Because a customer can have multiple orders, we use `SUM()`.

#### 2. Group the orders by month and customer

```sql
GROUP BY TO_CHAR(date, 'Month'), user_id
```

This produces one aggregated row for every **month-customer combination**.

Conceptually:

```text
May   → User 1 → Total = 965
May   → User 3 → Total = 860
May   → User 7 → Total = 740

June  → User 2 → Total = 1480
June  → User 4 → Total = 800
...
```

This `GROUP BY` is necessary because we want to rank customers according to their **total monthly spending**, rather than ranking individual orders.

#### 3. Partition the customers by month

```sql
PARTITION BY TO_CHAR(date, 'Month')
```

This creates a separate ranking window for every month.

Therefore:

```text
May
    User 1 → Rank 1
    User 3 → Rank 2
    User 7 → Rank 3

June
    User 2 → Rank 1
    User 4 → Rank 2
    ...

July
    User 5 → Rank 1
    User 2 → Rank 2
    ...
```

The ranking restarts from `1` for every month.

#### 4. Rank customers according to their total spending

```sql
ORDER BY SUM(amount) DESC
```

The customers with the highest total spending receive the highest ranking.

Because the ordering is descending:

```text
Highest spending → Rank 1
Second highest   → Rank 2
Third highest    → Rank 3
```

#### 5. Assign the ranking

```sql
RANK() OVER (
    PARTITION BY TO_CHAR(date, 'Month')
    ORDER BY SUM(amount) DESC
) AS month_rank
```

This is the core window-function operation.

It answers:

> Within each month, what is the rank of this customer based on their total spending?

#### 6. Use a subquery

```sql
SELECT *
FROM (
    ...
) AS t
```

The ranking is calculated inside the subquery.

The outer query can then use the generated `month_rank` column for filtering.

This is necessary because window functions are evaluated after the `WHERE` clause in SQL's logical query processing order. Therefore, we cannot generally filter a window-function result directly using `WHERE` at the same query level.

#### 7. Keep only the top two

```sql
WHERE t.month_rank < 3
```

This keeps:

```text
Rank 1
Rank 2
```

and removes:

```text
Rank 3+
```

Therefore, we obtain the top two ranked customers for every month.

#### 8. Sort the final result

```sql
ORDER BY month_name DESC, month_rank ASC;
```

The final result is ordered by month in descending order and rank in ascending order.

> **Important:** For production-quality SQL, ordering or grouping by formatted month names such as `'Month'` can be problematic because month names do not preserve chronological order and can behave unexpectedly across years. A date-based month key such as `DATE_TRUNC('month', date)` is generally more robust. If the analysis spans multiple years, this distinction becomes particularly important.

### Creating a Roll Number from Branch and Marks

Another practical use of `ROW_NUMBER()` is generating a group-wise sequence.

For example:

```sql
SELECT *,
       CONCAT(
           branch,
           '-',
           ROW_NUMBER() OVER(
               PARTITION BY branch
           )
       ) AS roll_no
FROM marks;
```

This generates identifiers such as:

```text
CSE-1
CSE-2
CSE-3
CSE-4

ECE-1
ECE-2
ECE-3
ECE-4

EEE-1
EEE-2
EEE-3
EEE-4
```

However, because no ordering criterion has been specified, the assignment of numbers within each branch should not be considered deterministic.

If the roll numbers should follow marks, for example, we can explicitly define the order:

```sql
SELECT *,
       CONCAT(
           branch,
           '-',
           ROW_NUMBER() OVER(
               PARTITION BY branch
               ORDER BY marks DESC
           )
       ) AS roll_no
FROM marks;
```

Now the highest-scoring student in each branch receives the first number:

```text
CSE → Highest marks → CSE-1
CSE → Second highest → CSE-2
...

ECE → Highest marks → ECE-1
...
```

If `student_id` should determine the order instead:

```sql
SELECT *,
       CONCAT(
           branch,
           '-',
           ROW_NUMBER() OVER(
               PARTITION BY branch
               ORDER BY student_id
           )
       ) AS roll_no
FROM marks;
```

Thus, `ROW_NUMBER()` is not merely a ranking function. It can also be used to create **deterministic, group-wise sequential identifiers** when combined with an appropriate `ORDER BY` clause.

### Ranking Functions at a Glance

| Function       | Purpose                          | Ties              | Gaps after ties? |
| :------------- | :------------------------------- | :---------------- | :--------------- |
| `ROW_NUMBER()` | Assign unique sequential numbers | Different numbers | No               |
| `RANK()`       | Competition-style ranking        | Same rank         | Yes              |
| `DENSE_RANK()` | Consecutive ranking              | Same rank         | No               |

A useful mental model is:

```text
Marks:        100   90   90   80

ROW_NUMBER:     1    2    3    4
RANK:           1    2    2    4
DENSE_RANK:     1    2    2    3
```

> **ROW_NUMBER() answers:** "What is this row's unique position?"
> **RANK() answers:** "What position does this value occupy, accounting for ties?"
> **DENSE_RANK() answers:** "What is this value's consecutive rank among distinct values?"

Your direction is correct, but there is one **very important conceptual issue** to fix before putting this into your notes:

`FIRST_VALUE()` and `NTH_VALUE()` are generally more intuitive for this topic, while `LAST_VALUE()` has a subtle interaction with the **window frame**. That makes this section particularly important for understanding how SQL window functions actually work.

Also, `NTH_VALUE()` without an explicit frame can produce results that appear surprising. I would therefore introduce the functions first, then frames, and only afterwards use them for branch-wise problems.

## FIRST_VALUE(), LAST_VALUE(), and NTH_VALUE()

These are **value window functions**. Unlike `RANK()` or `ROW_NUMBER()`, which return a position or rank, these functions return a **value from a particular row within the window**.

The three functions are:

| Function        | Purpose                                                  |
| :-------------- | :------------------------------------------------------- |
| `FIRST_VALUE()` | Returns the value from the first row of the window frame |
| `LAST_VALUE()`  | Returns the value from the last row of the window frame  |
| `NTH_VALUE()`   | Returns the value from the Nth row of the window frame   |

Their general syntax is:

```sql
FIRST_VALUE(expression) OVER (...)
LAST_VALUE(expression) OVER (...)
NTH_VALUE(expression, n) OVER (...)
```

For example:

```sql
FIRST_VALUE(name) OVER (
    ORDER BY marks DESC
)
```

means:

> Order the rows by marks from highest to lowest and return the `name` from the first row of the window.

---

### FIRST_VALUE()

`FIRST_VALUE()` returns the value of the specified expression from the **first row of the window frame**.

For example:

```sql
SELECT *,
       FIRST_VALUE(name) OVER (
           ORDER BY marks DESC
       ) AS topper
FROM marks;
```

Suppose the data is:

| name   | branch | marks |
| :----- | :----- | ----: |
| Deepak | CSE    |    98 |
| Arpan  | CSE    |    95 |
| Vinay  | ECE    |    95 |
| Rohit  | ECE    |    95 |
| Ankit  | ECE    |    88 |

Because the ordering is:

```sql
ORDER BY marks DESC
```

the first row has the highest marks.

Therefore, `FIRST_VALUE(name)` returns the name of the highest-scoring student.

Conceptually:

```text
Deepak → 98  ← First row
Arpan  → 95
Vinay  → 95
Rohit  → 95
Ankit  → 88
```

The result would therefore contain:

| name   | branch | marks | topper |
| :----- | :----- | ----: | :----- |
| Deepak | CSE    |    98 | Deepak |
| Arpan  | CSE    |    95 | Deepak |
| Vinay  | ECE    |    95 | Deepak |
| Rohit  | ECE    |    95 | Deepak |
| Ankit  | ECE    |    88 | Deepak |

Notice that the same value is returned for every row because the window is currently the **entire result set**.

### Finding the Branch Topper

To find the topper separately for each branch, use `PARTITION BY`:

```sql
SELECT *,
       FIRST_VALUE(name) OVER (
           PARTITION BY branch
           ORDER BY marks DESC
       ) AS branch_topper
FROM marks;
```

Now each branch forms an independent window.

Conceptually:

```text
CSE
98 → Deepak ← First row
95 → Arpan
78 → Shubham

ECE
95 → Vinay ← First row
95 → Rohit
88 → Ankit

EEE
91 → Rishabh ← First row
82 → Nitish
69 → Anukant
```

The result would look like:

| name    | branch | marks | branch_topper |
| :------ | :----- | ----: | :------------ |
| Deepak  | CSE    |    98 | Deepak        |
| Arpan   | CSE    |    95 | Deepak        |
| Shubham | CSE    |    78 | Deepak        |
| Vinay   | ECE    |    95 | Vinay         |
| Rohit   | ECE    |    95 | Vinay         |
| Ankit   | ECE    |    88 | Vinay         |
| Rishabh | EEE    |    91 | Rishabh       |
| Nitish  | EEE    |    82 | Rishabh       |
| Anukant | EEE    |    69 | Rishabh       |

Thus:

```sql
PARTITION BY branch
```

makes the function operate independently within every branch.

---

### LAST_VALUE()

`LAST_VALUE()` returns the value from the **last row of the window frame**.

This sounds straightforward, but `LAST_VALUE()` is one of the window functions where understanding the **frame** is extremely important.

Consider:

```sql
SELECT *,
       LAST_VALUE(name) OVER (
           ORDER BY marks DESC
       ) AS last_value
FROM marks;
```

A common mistake is to assume that this will always return the student with the lowest marks.

It does not necessarily do so.

The reason is the **default window frame**.

With an ordered window, the frame commonly ends at the current row. Therefore, for a given row, the "last row" of the frame may simply be the **current row itself**.

Consequently, `LAST_VALUE()` can appear to simply return the current row's value.

To explicitly obtain the last row of the entire partition, we should define the frame:

```sql
SELECT *,
       LAST_VALUE(name) OVER (
           ORDER BY marks DESC
           ROWS BETWEEN UNBOUNDED PRECEDING
                    AND UNBOUNDED FOLLOWING
       ) AS lowest_scorer
FROM marks;
```

Now the frame explicitly extends from the first row to the last row of the entire window.

Since the ordering is descending:

```text
98
95
95
88
80
```

the last row is the student with `80`.

Therefore, `LAST_VALUE(name)` returns the lowest-scoring student's name.

### Finding the Last Student of Each Branch

We can combine `LAST_VALUE()` with `PARTITION BY`:

```sql
SELECT *,
       LAST_VALUE(name) OVER (
           PARTITION BY branch
           ORDER BY marks DESC
           ROWS BETWEEN UNBOUNDED PRECEDING
                    AND UNBOUNDED FOLLOWING
       ) AS last_student
FROM marks;
```

The important part is:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING
         AND UNBOUNDED FOLLOWING
```

Without this frame, `LAST_VALUE()` may not give the result you expect.

If we want the student with the **lowest marks** in each branch, another way is to reverse the ordering and use `FIRST_VALUE()`:

```sql
SELECT *,
       FIRST_VALUE(name) OVER (
           PARTITION BY branch
           ORDER BY marks ASC
       ) AS lowest_scorer
FROM marks;
```

This is often easier to understand.

> **Practical rule:** When you want the last value of an entire partition, be especially careful with the window frame. `FIRST_VALUE()` with reversed ordering is often a simpler alternative.

---

### NTH_VALUE()

`NTH_VALUE()` returns the value from the **Nth row of the window frame**.

Syntax:

```sql
NTH_VALUE(expression, n) OVER (...)
```

Here:

* `expression` is the column/value we want to retrieve.
* `n` specifies which row to retrieve.
* `OVER()` defines the window.
* `ORDER BY` determines the position of rows.
* `PARTITION BY` optionally creates separate windows.
* The frame determines which rows are available to the function.

For example:

```sql
SELECT *,
       NTH_VALUE(name, 1) OVER (
           ORDER BY marks DESC
       ) AS first_student
FROM marks;
```

`NTH_VALUE(name, 1)` means:

> Return the `name` from the first row of the window.

Therefore, this is conceptually similar to:

```sql
FIRST_VALUE(name) OVER (
    ORDER BY marks DESC
)
```

Likewise:

```sql
NTH_VALUE(name, 2)
```

returns the value from the second row.

And:

```sql
NTH_VALUE(name, 5)
```

returns the value from the fifth row.

### Finding the Topper Using NTH_VALUE()

```sql
SELECT *,
       NTH_VALUE(name, 1) OVER (
           ORDER BY marks DESC
       ) AS topper
FROM marks;
```

Because the ordering is descending:

```text
98 → 1st
95 → 2nd
91 → 3rd
88 → 4th
80 → 5th
```

`NTH_VALUE(name, 1)` returns the name of the student with the highest marks.

### Finding the Lowest-Scoring Student Using NTH_VALUE()

We can reverse the ordering:

```sql
SELECT *,
       NTH_VALUE(name, 1) OVER (
           ORDER BY marks ASC
       ) AS lowest_scorer
FROM marks;
```

Now:

```text
Lowest marks → First row
...
Highest marks → Last row
```

Therefore, `NTH_VALUE(name, 1)` gives the student with the lowest marks.

However, when using `NTH_VALUE()` to retrieve a value from the entire partition, explicitly specifying the frame is often preferable:

```sql
SELECT *,
       NTH_VALUE(name, 1) OVER (
           ORDER BY marks ASC
           ROWS BETWEEN UNBOUNDED PRECEDING
                    AND UNBOUNDED FOLLOWING
       ) AS lowest_scorer
FROM marks;
```

This removes ambiguity about the scope of the frame.

---

## FRAME Clause

A **window frame** is a subset of rows within a window that defines exactly which rows are considered by a window function for the current row.

This distinction is important:

> **Window = the complete set of rows available to the window function.**
> **Frame = the specific subset of that window currently considered by the function.**

For example:

```sql
PARTITION BY branch
```

may create a window containing all students of the CSE branch.

A frame can then specify which portion of those CSE students should be considered for the current row.

### Basic Structure

A frame can be written as:

```sql
ROWS BETWEEN <starting_boundary>
         AND <ending_boundary>
```

For example:

```sql
ROWS BETWEEN 3 PRECEDING
         AND 2 FOLLOWING
```

The frame is defined relative to the **current row**.

There are two important concepts:

* `ROWS` defines the frame in terms of physical rows.
* `BETWEEN ... AND ...` defines the starting and ending boundaries.

### Understanding `ROWS`

`ROWS` tells SQL that the frame should be based on the actual rows in the ordered window.

For example:

```sql
ROWS BETWEEN 3 PRECEDING AND 2 FOLLOWING
```

means:

```text
3 rows before current row
        ↓
Current row
        ↓
2 rows after current row
```

So the frame can contain up to:

```text
3 + 1 + 2 = 6 rows
```

The number can be smaller near the beginning or end of the partition because there may not be enough preceding or following rows.

---

## Understanding Frame Boundaries

### `UNBOUNDED PRECEDING`

```sql
UNBOUNDED PRECEDING
```

means:

> Start from the very first row of the partition.

It does not mean "an unlimited number of rows before the current row." It specifically refers to the **beginning of the partition**.

### `UNBOUNDED FOLLOWING`

```sql
UNBOUNDED FOLLOWING
```

means:

> Extend the frame all the way to the final row of the partition.

### `CURRENT ROW`

```sql
CURRENT ROW
```

means:

> The row currently being evaluated.

### `n PRECEDING`

For example:

```sql
3 PRECEDING
```

means:

> Three rows before the current row.

### `n FOLLOWING`

For example:

```sql
2 FOLLOWING
```

means:

> Two rows after the current row.

---

## Common Frame Definitions

### 1. From the Beginning to the Current Row

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
```

Breakdown:

```text
ROWS
```

Use physical rows to define the frame.

```text
BETWEEN
```

Specify both the starting and ending boundaries.

```text
UNBOUNDED PRECEDING
```

Start from the first row of the partition.

```text
AND
```

Connect the starting and ending boundaries.

```text
CURRENT ROW
```

End at the current row.

Therefore:

```text
[First row] ─────────────── [Current row]
```

This is commonly used for **running calculations**.

For example, a running total:

```sql
SELECT *,
       SUM(amount) OVER (
           ORDER BY date
           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
       ) AS running_total
FROM orders;
```

If the amounts are:

| date  | amount |
| :---- | -----: |
| Jan 1 |    100 |
| Jan 2 |    200 |
| Jan 3 |    150 |
| Jan 4 |    300 |

the running total becomes:

| date  | amount | running_total |
| :---- | -----: | ------------: |
| Jan 1 |    100 |           100 |
| Jan 2 |    200 |           300 |
| Jan 3 |    150 |           450 |
| Jan 4 |    300 |           750 |

This is one of the most important applications of window frames.

---

### 2. One Row Before to One Row After

```sql
ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
```

This means:

```text
Previous row
     ↓
Current row
     ↓
Next row
```

For example:

```sql
SELECT *,
       AVG(marks) OVER (
           ORDER BY student_id
           ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
       ) AS moving_avg
FROM marks;
```

This calculates an average using the current row, the preceding row, and the following row.

For:

```text
80
90
70
60
```

the frame for `90` is:

```text
80, 90, 70
```

and the average is:

```text
(80 + 90 + 70) / 3 = 80
```

This type of frame is useful for **moving averages, local comparisons, smoothing, and trend analysis**.

> Note: At the first and last rows, the frame naturally contains fewer rows because a preceding or following row does not exist.

---

### 3. Entire Partition

```sql
ROWS BETWEEN UNBOUNDED PRECEDING
         AND UNBOUNDED FOLLOWING
```

This means:

```text
[First row] ───────────────────── [Last row]
```

The frame contains **every row in the partition**, regardless of the current row.

This is particularly useful with `LAST_VALUE()` and `NTH_VALUE()` when we want to retrieve a value from anywhere in the complete partition.

For example:

```sql
SELECT *,
       LAST_VALUE(name) OVER (
           PARTITION BY branch
           ORDER BY marks DESC
           ROWS BETWEEN UNBOUNDED PRECEDING
                    AND UNBOUNDED FOLLOWING
       ) AS lowest_scorer
FROM marks;
```

Here, the complete branch is available to `LAST_VALUE()`.

---

### 4. Three Rows Before to Two Rows After

```sql
ROWS BETWEEN 3 PRECEDING AND 2 FOLLOWING
```

This means:

```text
3 preceding rows
        ↓
2 preceding rows
        ↓
1 preceding row
        ↓
Current row
        ↓
1 following row
        ↓
2 following rows
```

Therefore, the frame can contain up to six rows:

```text
3 + 1 + 2 = 6
```

For example:

```sql
SELECT *,
       AVG(marks) OVER (
           ORDER BY student_id
           ROWS BETWEEN 3 PRECEDING AND 2 FOLLOWING
       ) AS local_average
FROM marks;
```

This can be useful when performing a wider **moving-window analysis**, where the current row is evaluated in the context of nearby observations.

---

## Finding the Lowest-Scoring Student of Each Branch

We can use `NTH_VALUE()` to retrieve the first student after ordering marks in ascending order.

```sql
SELECT *,
       NTH_VALUE(marks, 1) OVER (
           PARTITION BY branch
           ORDER BY marks ASC
           ROWS BETWEEN UNBOUNDED PRECEDING
                    AND UNBOUNDED FOLLOWING
       ) AS lowest_marks
FROM marks;
```

Breakdown:

```sql
PARTITION BY branch
```

Creates a separate window for every branch.

```sql
ORDER BY marks ASC
```

Places the lowest marks at the beginning of every branch.

```sql
NTH_VALUE(marks, 1)
```

Retrieves the value from the first row.

```sql
ROWS BETWEEN UNBOUNDED PRECEDING
         AND UNBOUNDED FOLLOWING
```

Makes the entire branch available as the frame.

Therefore:

> The first row of each branch is its lowest-scoring student, so `NTH_VALUE(marks, 1)` returns the lowest marks for that branch.

---

## Finding the Second Topper of Each Branch

To find the second-highest marks, arrange the rows in descending order and retrieve the second value:

```sql
SELECT *,
       NTH_VALUE(marks, 2) OVER (
           PARTITION BY branch
           ORDER BY marks DESC
           ROWS BETWEEN UNBOUNDED PRECEDING
                    AND UNBOUNDED FOLLOWING
       ) AS second_highest_marks
FROM marks;
```

The ordering becomes:

```text
98 → 1st
95 → 2nd
78 → 3rd
43 → 4th
```

Therefore:

```text
NTH_VALUE(marks, 2)
```

returns `95`.

### Finding the 5th Topper

The same principle can be extended to any position:

```sql
NTH_VALUE(name, 5) OVER (
    PARTITION BY branch
    ORDER BY marks DESC
    ROWS BETWEEN UNBOUNDED PRECEDING
             AND UNBOUNDED FOLLOWING
)
```

This returns the name of the **5th row after ordering students by marks in descending order within each branch**.

An important distinction:

> `NTH_VALUE()` identifies the Nth **row**, not necessarily the Nth distinct mark.

If two students are tied, they occupy separate rows.

For example:

```text
Marks: 98, 95, 95, 88, 80

1st row → 98
2nd row → 95
3rd row → 95
4th row → 88
5th row → 80
```

If the requirement is specifically the **5th distinct score**, `NTH_VALUE()` alone is not sufficient; the problem should first be transformed into distinct rankings using techniques such as `DENSE_RANK()`.

---

## Fetching the Complete Details of the Branch Toppers

Suppose we want not just the topper's marks but the **complete row** of the topper.

We can first calculate the topper's name and marks and then filter the result using a CTE.

```sql
WITH t AS (
    SELECT *,
           NTH_VALUE(name, 1) OVER (
               PARTITION BY branch
               ORDER BY marks DESC
           ) AS topper_name,
           NTH_VALUE(marks, 1) OVER (
               PARTITION BY branch
               ORDER BY marks DESC
           ) AS topper_marks
    FROM marks
)
SELECT *
FROM t
WHERE t.name = t.topper_name
  AND t.marks = t.topper_marks;
```

Conceptually, the CTE creates additional analytical columns:

| name    | branch | marks | topper_name | topper_marks |
| :------ | :----- | ----: | :---------- | -----------: |
| Deepak  | CSE    |    98 | Deepak      |           98 |
| Arpan   | CSE    |    95 | Deepak      |           98 |
| Shubham | CSE    |    78 | Deepak      |           98 |
| Vinay   | ECE    |    95 | Vinay       |           95 |
| Rohit   | ECE    |    95 | Vinay       |           95 |
| Ankit   | ECE    |    88 | Vinay       |           95 |

The outer query then keeps only rows where:

```sql
name = topper_name
```

and:

```sql
marks = topper_marks
```

This returns the actual topper rows.

### An Important Tie Consideration

There is a subtle issue with this approach.

Suppose two students have the highest marks:

```text
Vinay  → 95
Rohit  → 95
Ankit  → 88
```

`NTH_VALUE(name, 1)` will return only the name of whichever tied row appears first according to the ordering.

Therefore, the condition:

```sql
name = topper_name
```

may return only one of the tied toppers.

If the requirement is:

> Find **all students tied for the highest marks** in each branch.

then `RANK()` or `DENSE_RANK()` is generally more appropriate:

```sql
WITH ranked_students AS (
    SELECT *,
           RANK() OVER (
               PARTITION BY branch
               ORDER BY marks DESC
           ) AS branch_rank
    FROM marks
)
SELECT *
FROM ranked_students
WHERE branch_rank = 1;
```

This is an important practical distinction between **finding the first row** and **finding all rows sharing the highest value**.

---

## Alternative Way of Writing Window Functions

When the same window definition is used multiple times, SQL allows us to define the window once using the `WINDOW` clause.

Instead of repeatedly writing:

```sql
PARTITION BY branch
ORDER BY marks ASC
ROWS BETWEEN UNBOUNDED PRECEDING
         AND UNBOUNDED FOLLOWING
```

we can define it once:

```sql
WINDOW w AS (
    PARTITION BY branch
    ORDER BY marks ASC
    ROWS BETWEEN UNBOUNDED PRECEDING
             AND UNBOUNDED FOLLOWING
)
```

and then reuse it.

For example:

```sql
WITH bottom_studs AS (
    SELECT *,
           NTH_VALUE(name, 1) OVER w AS bottom_name,
           NTH_VALUE(marks, 1) OVER w AS bottom_marks
    FROM marks
    WINDOW w AS (
        PARTITION BY branch
        ORDER BY marks ASC
        ROWS BETWEEN UNBOUNDED PRECEDING
                 AND UNBOUNDED FOLLOWING
    )
)
SELECT name, branch, marks
FROM bottom_studs
WHERE bottom_studs.name = bottom_studs.bottom_name
  AND bottom_studs.marks = bottom_studs.bottom_marks;
```

### Understanding the `WINDOW` Clause

The following:

```sql
WINDOW w AS (
    PARTITION BY branch
    ORDER BY marks ASC
    ROWS BETWEEN UNBOUNDED PRECEDING
             AND UNBOUNDED FOLLOWING
)
```

creates a named window called `w`.

We can then write:

```sql
NTH_VALUE(name, 1) OVER w
```

instead of repeatedly writing the complete window specification.

Thus:

```sql
OVER w
```

is effectively shorthand for:

```sql
OVER (
    PARTITION BY branch
    ORDER BY marks ASC
    ROWS BETWEEN UNBOUNDED PRECEDING
             AND UNBOUNDED FOLLOWING
)
```

This improves **readability, consistency, and maintainability**, especially when multiple window functions use the same partitioning, ordering, and frame definition.

### Why This Is Useful

Consider:

```sql
NTH_VALUE(name, 1) OVER w
NTH_VALUE(marks, 1) OVER w
FIRST_VALUE(name) OVER w
LAST_VALUE(name) OVER w
```

All four functions can operate over exactly the same window definition.

Instead of duplicating the specification four times, we define it once.

This becomes particularly valuable in analytical queries containing many window functions.

---

## FIRST_VALUE() vs. LAST_VALUE() vs. NTH_VALUE()

| Function        | Returns                  | Example              |
| :-------------- | :----------------------- | :------------------- |
| `FIRST_VALUE()` | First value in the frame | First/topper student |
| `LAST_VALUE()`  | Last value in the frame  | Last/lowest student  |
| `NTH_VALUE()`   | Nth value in the frame   | 2nd or 5th student   |

A useful mental model is:

```text
ORDER BY marks DESC

98   ← FIRST_VALUE()
95
91   ← NTH_VALUE(..., 3)
88
80   ← LAST_VALUE()
```

The crucial point is that these functions operate on the **window frame**, not simply on the entire partition by default.

Therefore, when working with `LAST_VALUE()` or `NTH_VALUE()`, always ask:

> **"What exactly is my frame?"**

That question prevents many of the most common mistakes with value window functions.

## LEAD() and LAG()

`LEAD()` and `LAG()` are **value window functions** used to access values from another row relative to the current row.

They are particularly useful for **time-series analysis, trend analysis, period-over-period comparisons, change detection, and sequential data analysis**.

The key difference is:

* `LAG()` looks **backward** from the current row.
* `LEAD()` looks **forward** from the current row.

### LAG()

`LAG()` retrieves a value from a row that occurs **before the current row** according to the ordering specified in the window.

Basic syntax:

```sql
LAG(expression, offset, default_value)
OVER (
    ORDER BY column
)
```

Where:

* `expression` is the column whose value we want.
* `offset` specifies how many rows to look backward.
* `default_value` is the value returned when the requested preceding row does not exist.

The `offset` and `default_value` are optional.

Therefore:

```sql
LAG(marks)
```

is equivalent to:

```sql
LAG(marks, 1)
```

and means:

> Retrieve the value of `marks` from the immediately preceding row.

For example:

```sql
SELECT *,
       LAG(marks) OVER (
           ORDER BY student_id
       ) AS previous_marks
FROM marks;
```

Suppose:

| student_id | name    | marks |
| :--------- | :------ | ----: |
| 1          | Nitish  |    82 |
| 2          | Rishabh |    91 |
| 3          | Anukant |    69 |
| 4          | Rupesh  |    55 |

The result would be:

| student_id | name    | marks | previous_marks |
| :--------- | :------ | ----: | -------------: |
| 1          | Nitish  |    82 |           NULL |
| 2          | Rishabh |    91 |             82 |
| 3          | Anukant |    69 |             91 |
| 4          | Rupesh  |    55 |             69 |

The first row has no preceding row, so `LAG()` returns `NULL`.

Conceptually:

```text
Current Row       LAG()
    ↓               ↓
Rishabh (91) ← Nitish (82)
Anukant (69) ← Rishabh (91)
Rupesh (55)  ← Anukant (69)
```

### LAG() with an Offset

We can specify how many rows to move backward.

```sql
LAG(marks, 2) OVER (
    ORDER BY student_id
)
```

This means:

> Retrieve the marks from two rows before the current row.

For example:

| student_id | marks | LAG(marks, 2) |
| :--------- | ----: | ------------: |
| 1          |    82 |          NULL |
| 2          |    91 |          NULL |
| 3          |    69 |            82 |
| 4          |    55 |            91 |

The first two rows have no second-preceding row, so they return `NULL`.

### LEAD()

`LEAD()` is the opposite of `LAG()`.

It retrieves a value from a row that occurs **after the current row** according to the specified ordering.

Syntax:

```sql
LEAD(expression, offset, default_value)
OVER (
    ORDER BY column
)
```

For example:

```sql
SELECT *,
       LEAD(marks) OVER (
           ORDER BY student_id
       ) AS next_marks
FROM marks;
```

Result:

| student_id | name    | marks | next_marks |
| :--------- | :------ | ----: | ---------: |
| 1          | Nitish  |    82 |         91 |
| 2          | Rishabh |    91 |         69 |
| 3          | Anukant |    69 |         55 |
| 4          | Rupesh  |    55 |       NULL |

Conceptually:

```text
Current Row       LEAD()
    ↓               ↓
Nitish (82)   → Rishabh (91)
Rishabh (91)  → Anukant (69)
Anukant (69)  → Rupesh (55)
Rupesh (55)   → NULL
```

Thus:

> `LAG()` looks backward, whereas `LEAD()` looks forward.

### LAG() vs LEAD()

| Function          | Direction         | Common use               |
| :---------------- | :---------------- | :----------------------- |
| `LAG()`           | Previous row      | Previous month's revenue |
| `LEAD()`          | Next row          | Next month's revenue     |
| `LAG(column, 2)`  | Two rows backward | Value two periods ago    |
| `LEAD(column, 2)` | Two rows forward  | Value two periods ahead  |

---

## Practical Applications

`LEAD()` and `LAG()` are particularly useful when the value of the **current row needs to be compared with another row**.

Common applications include:

* Month-over-month revenue growth
* Year-over-year revenue growth
* Comparing current and previous salaries
* Detecting changes in status
* Finding the time between consecutive events
* Comparing current and previous prices
* Identifying increases or decreases in metrics
* Calculating retention and churn-related metrics
* Time-series analysis

For example, suppose monthly revenue is:

| Month    | Revenue |
| :------- | ------: |
| January  |    1000 |
| February |    1200 |
| March    |    1500 |
| April    |    1300 |

Using `LAG()`:

| Month    | Revenue | Previous Revenue |
| :------- | ------: | ---------------: |
| January  |    1000 |             NULL |
| February |    1200 |             1000 |
| March    |    1500 |             1200 |
| April    |    1300 |             1500 |

We can then calculate:

```text
Revenue Difference = Current Revenue - Previous Revenue
```

and:

```text
Revenue % Change =
(Current Revenue - Previous Revenue) / Previous Revenue × 100
```

This is exactly what makes `LAG()` so useful for **period-over-period analysis**.

# Month-over-Month Revenue Growth

Suppose we have an `orders` table containing:

```text
date
user_id
amount
```

and we want to answer:

> What was Zomato's month-over-month revenue growth?

The problem can be divided into three stages:

```text
Orders
   ↓
Monthly Revenue
   ↓
Previous Month's Revenue
   ↓
Revenue Difference / Percentage Change
```

### Step 1: Calculate Monthly Revenue

First, we need to aggregate individual orders into monthly revenue.

A robust approach is:

```sql
SELECT
    DATE_TRUNC('month', date) AS month_date,
    SUM(amount) AS revenue
FROM orders
GROUP BY DATE_TRUNC('month', date);
```

For example:

| month_date | revenue |
| :--------- | ------: |
| 2026-01-01 |  100000 |
| 2026-02-01 |  125000 |
| 2026-03-01 |  150000 |
| 2026-04-01 |  135000 |

At this point, we have one row per month.

### Step 2: Retrieve the Previous Month's Revenue

Now we can use `LAG()`:

```sql
SELECT
    month_date,
    revenue,
    LAG(revenue) OVER (
        ORDER BY month_date
    ) AS previous_revenue
FROM monthly_revenue;
```

This produces:

| month_date | revenue | previous_revenue |
| :--------- | ------: | ---------------: |
| Jan        |  100000 |             NULL |
| Feb        |  125000 |           100000 |
| Mar        |  150000 |           125000 |
| Apr        |  135000 |           150000 |

`LAG()` therefore gives us the value required for the comparison.

### Step 3: Calculate Revenue Difference

The absolute change is:

```sql
revenue - previous_revenue
```

For example:

```text
February:
125000 - 100000 = 25000

March:
150000 - 125000 = 25000

April:
135000 - 150000 = -15000
```

A positive value indicates growth, while a negative value indicates a decline.

### Step 4: Calculate Percentage Change

The standard formula is:

```text
Percentage Change =
(Current - Previous) / Previous × 100
```

Therefore:

```sql
(
    (revenue - previous_revenue)
    / previous_revenue
) * 100
```

For example, if January revenue is `100000` and February revenue is `125000`:

```text
(125000 - 100000) / 100000 × 100
= 25%
```

Therefore, February experienced **25% month-over-month revenue growth**.

---

## Complete Query

A clean and production-oriented implementation is:

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', date) AS month_date,
        TO_CHAR(date, 'FMMonth') AS month,
        SUM(amount) AS revenue
    FROM orders
    GROUP BY
        DATE_TRUNC('month', date),
        TO_CHAR(date, 'FMMonth')
),
revenue_with_lag AS (
    SELECT
        month_date,
        month,
        revenue,
        LAG(revenue) OVER (
            ORDER BY month_date
        ) AS previous_revenue
    FROM monthly_revenue
)
SELECT
    month,
    revenue,
    previous_revenue,
    revenue - previous_revenue AS revenue_diff,
    (
        (revenue - previous_revenue)
        / NULLIF(previous_revenue, 0)
    ) * 100 AS revenue_pct_change
FROM revenue_with_lag
ORDER BY month_date;
```

### Query Breakdown

#### 1. Create Monthly Revenue

```sql
WITH monthly_revenue AS (
```

A CTE named `monthly_revenue` is created.

The purpose of this CTE is to transform the raw order-level data into **month-level aggregated data**.

#### 2. Create a Proper Month Identifier

```sql
DATE_TRUNC('month', date) AS month_date
```

`DATE_TRUNC('month', date)` converts a date into the beginning of its month.

For example:

```text
2026-01-17 → 2026-01-01
2026-01-29 → 2026-01-01
2026-02-15 → 2026-02-01
```

This gives us a reliable value for chronological ordering.

This is preferable to ordering by:

```sql
TO_CHAR(date, 'Month')
```

because alphabetical ordering of month names does not represent chronological order.

#### 3. Create a Display-Friendly Month Name

```sql
TO_CHAR(date, 'FMMonth') AS month
```

This converts the date into a human-readable month name.

The `FM` modifier removes unnecessary padding spaces produced by PostgreSQL's formatting behaviour.

Therefore, instead of:

```text
January····
February···
```

we get:

```text
January
February
```

The important distinction is:

```text
month_date → used for computation and sorting
month      → used for presentation
```

#### 4. Calculate Revenue

```sql
SUM(amount) AS revenue
```

All orders belonging to the same month are summed together.

#### 5. Group the Data

```sql
GROUP BY
    DATE_TRUNC('month', date),
    TO_CHAR(date, 'FMMonth')
```

This produces one aggregated record per month.

The resulting CTE conceptually looks like:

| month_date | month    | revenue |
| :--------- | :------- | ------: |
| 2026-01-01 | January  |  100000 |
| 2026-02-01 | February |  125000 |
| 2026-03-01 | March    |  150000 |

---

### 5. Calculate Previous Month's Revenue

The second CTE is:

```sql
revenue_with_lag AS (
```

Inside it:

```sql
LAG(revenue) OVER (
    ORDER BY month_date
) AS previous_revenue
```

The months are first ordered chronologically:

```text
January
February
March
April
```

Then `LAG()` looks one row backward.

Therefore:

```text
January  → NULL
February → January revenue
March    → February revenue
April    → March revenue
```

---

### 6. Calculate Revenue Difference

```sql
revenue - previous_revenue AS revenue_diff
```

This calculates the absolute change in revenue.

```text
Current Revenue - Previous Revenue
```

A positive value means revenue increased.

A negative value means revenue decreased.

---

### 7. Calculate Revenue Percentage Change

```sql
(
    (revenue - previous_revenue)
    / NULLIF(previous_revenue, 0)
) * 100 AS revenue_pct_change
```

This calculates the MoM percentage change.

`NULLIF(previous_revenue, 0)` is particularly important.

If the previous month's revenue is `0`, then:

```text
current / 0
```

would result in a division-by-zero error.

`NULLIF(previous_revenue, 0)` converts `0` into `NULL`, preventing that error.

Therefore:

```text
NULLIF(0, 0) → NULL
```

and division by `NULL` produces `NULL` rather than an error.

---

## A Simpler Version

Once the concept is understood, the same logic can be written more compactly:

```sql
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', date) AS month_date,
        SUM(amount) AS revenue
    FROM orders
    GROUP BY DATE_TRUNC('month', date)
),
revenue_with_lag AS (
    SELECT
        month_date,
        revenue,
        LAG(revenue) OVER (
            ORDER BY month_date
        ) AS previous_revenue
    FROM monthly_revenue
)
SELECT
    month_date,
    revenue,
    previous_revenue,
    revenue - previous_revenue AS revenue_diff,
    ROUND(
        100.0 * (revenue - previous_revenue)
        / NULLIF(previous_revenue, 0),
        2
    ) AS revenue_pct_change
FROM revenue_with_lag
ORDER BY month_date;
```

The `100.0` is intentional: depending on the data types involved, using `100` can lead to integer-division behaviour in some SQL systems. Explicitly using a decimal value makes the percentage calculation safer.

---

## Important Observation About the First Query

Your original query was:

```sql
SELECT TO_CHAR(date, 'Month') AS month,
       SUM(amount),
       LAG(SUM(amount)) OVER(
           ORDER BY TO_CHAR(date, 'Month') DESC
       ) AS lag,
       SUM(amount) - LAG(SUM(amount)) OVER(
           ORDER BY TO_CHAR(date, 'Month') DESC
       ) AS revenue_diff
FROM orders
GROUP BY TO_CHAR(date, 'Month')
ORDER BY TO_CHAR(date, 'Month') DESC;
```

This demonstrates an important SQL capability:

> A window function can operate on the result of an aggregate function.

Here:

```sql
SUM(amount)
```

first produces the monthly revenue, and:

```sql
LAG(SUM(amount)) OVER(...)
```

then compares those aggregated monthly values.

However, I would **not recommend this version for your final notes as the preferred production approach** because:

1. `TO_CHAR(date, 'Month')` loses the year.
2. Month names are not reliable chronological sorting keys.
3. It can produce incorrect results when data spans multiple years.
4. The expression is repeated multiple times.
5. The more explicit CTE approach makes the logical stages easier to understand and maintain.

The CTE version demonstrates a more important analytical SQL pattern:

```text
Raw data
   ↓
GROUP BY
   ↓
Aggregated data
   ↓
LAG()
   ↓
Comparison
   ↓
Business metric
```

This pattern appears frequently in real-world data analyst and business intelligence work.

### LEAD() — The Opposite of LAG()

The same monthly revenue dataset can be analysed using `LEAD()`.

```sql
SELECT
    month_date,
    revenue,
    LEAD(revenue) OVER (
        ORDER BY month_date
    ) AS next_month_revenue
FROM monthly_revenue;
```

Result:

| month    | revenue | next_month_revenue |
| :------- | ------: | -----------------: |
| January  |  100000 |             125000 |
| February |  125000 |             150000 |
| March    |  150000 |             135000 |
| April    |  135000 |               NULL |

`LEAD()` looks **forward**, whereas `LAG()` looks **backward**.

This makes `LEAD()` useful when the question involves a future event or the next observation.

For example:

> "What will the next month's revenue be?"

```sql
LEAD(revenue)
```

or:

> "How many days until the customer's next order?"

A combination of `LEAD()` and date arithmetic can answer that directly.

### Mental Model

Keep this simple rule in mind:

```text
                Current Row
                    │
          ┌─────────┴─────────┐
          ↓                   ↓
       LAG()                LEAD()
     looks back            looks ahead
```

Or, in analytical terms:

```text
Previous Period ──→ Current Period ──→ Next Period
       ↑                 ↑                 ↑
     LAG()             Current           LEAD()
```

This makes `LAG()` particularly suited to **MoM, QoQ, and YoY comparisons**, while `LEAD()` is useful when analysing **future or subsequent observations**.